In [ ]:
import gym
from gym import spaces
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

class ALMEnv(gym.Env):
    def __init__(self, rho_LB=0.70):  # Default ρLB = 0.70
        super(ALMEnv, self).__init__()

        # Define observation and action spaces
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0, -1.0, -1.0, -1.0, 0]),
            high=np.array([5.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 30]),
            dtype=np.float32
        )
        self.action_space = spaces.Box(low=np.array([0.0, 0.0, 0.0]), high=np.array([1.0, 1.0, 1.0]), dtype=np.float32)

        # Parameters
        self.mu_B, self.mu_S, self.mu_L = 0.0651, 0.1175, 0.1330  
        self.sigma_B, self.sigma_S, self.sigma_L = 0.0969, 0.1439, 0.0682  
        self.rho_SB, self.rho_LS = 0.25, 0.34  
        self.rho_LB = rho_LB  # Correlation between liabilities and bonds (ρLB)
        self.risk_free_rate = 0.0501  
        self.time_horizon = 30  
        self.gamma = 4  # Fixed risk aversion (γ = 4)

        self.reset()

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)

        # Initial portfolio allocations
        self.Funding_Ratio = np.random.uniform(0.8, 1.2)  
        self.alloc_stock, self.alloc_bond = 0.4, 0.5  
        self.alloc_cash = 1.0 - (self.alloc_stock + self.alloc_bond)  
        self.time_step = 0  
        self.r = self.risk_free_rate  

        observation = np.array([
            max(self.Funding_Ratio, 1e-5),  
            self.alloc_stock, self.alloc_bond, self.alloc_cash,
            self.rho_SB, self.rho_LB, self.rho_LS,
            self.time_step
        ], dtype=np.float32)

        return observation, {}

    def vasicek_model(self, r, kappa=0.1, theta=0.05, sigma_r=0.02, dt=1/30):
        dr = kappa * (theta - r) * dt + sigma_r * np.random.normal(0, np.sqrt(dt))
        return r + dr

    def step(self, action):
        action = action / np.sum(action) if np.sum(action) != 0 else np.array([1/3, 1/3, 1/3])
        self.alloc_stock, self.alloc_bond, self.alloc_cash = action

        self.r = self.vasicek_model(self.r)

        stock_return = np.random.normal(self.mu_S, self.sigma_S)
        bond_return = np.random.normal(self.mu_B, self.sigma_B)
        cash_return = self.r  

        portfolio_return = (self.alloc_stock * stock_return +
                            self.alloc_bond * bond_return +
                            self.alloc_cash * cash_return)

        liabilities_growth = np.random.normal(self.mu_L, self.sigma_L)
        self.Funding_Ratio *= (1 + portfolio_return) / (1 + liabilities_growth)

        # Compute theoretical PSP and LHP values
        lambda_S = (self.mu_S - self.r) / self.sigma_S
        lambda_B = (self.mu_B - self.r) / self.sigma_B

        pi_PSP_S = lambda_S / (self.gamma * self.sigma_S * np.sqrt(1 - self.rho_SB**2))
        pi_LHP_S = (1 - 1 / self.gamma) * (self.sigma_L * (self.rho_LS - self.rho_LB * self.rho_SB)) / (self.sigma_S * (1 - self.rho_SB**2))
        
        pi_PSP_B = (1 / (self.gamma * self.sigma_B)) * (lambda_B - lambda_S * (self.rho_SB / np.sqrt(1 - self.rho_SB**2)))
        pi_LHP_B = (1 - 1 / self.gamma) * (self.sigma_L * (self.rho_LB - self.rho_LS * self.rho_SB)) / (self.sigma_B * (1 - self.rho_SB**2))

        reward = -0.5 * (abs(self.alloc_stock - pi_PSP_S) + abs(self.alloc_stock - pi_LHP_S) +
                         abs(self.alloc_bond - pi_PSP_B) + abs(self.alloc_bond - pi_LHP_B))

        self.time_step += 1
        terminated = self.time_step >= self.time_horizon
        truncated = False

        next_state = np.array([
            max(self.Funding_Ratio, 1e-5),
            self.alloc_stock, self.alloc_bond, self.alloc_cash,
            self.rho_SB, self.rho_LB, self.rho_LS,
            self.time_step
        ], dtype=np.float32)

        return next_state, reward, terminated, truncated, {}

# Train the model
rho_LB_values = [-1.00,-0.70,-0.50,-0.30,0.00,0.30,0.50,0.70,1.00]  # Correlation (ρLB) values
results = []

for rho_LB in rho_LB_values:
    env = make_vec_env(lambda: ALMEnv(rho_LB), n_envs=1)
    model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="./alm_tensorboard/")
    model.learn(total_timesteps=100000)  
    model.save(f"ppo_alm_rho_LB_{rho_LB}")

    reset_result = env.reset()
    if isinstance(reset_result, tuple) and len(reset_result) == 2:
        obs, info = reset_result  # New Gym version (returns obs, info)
    else:
        obs = reset_result  # Old Gym version (returns only obs)

    done = False
    while not done:
        action, _ = model.predict(obs)
    
        step_result = env.step(action)
        if len(step_result) == 5:
           obs, reward, terminated, truncated, info = step_result
        else:
           obs, reward, done, info = step_result
           terminated, truncated = done, False

        done = terminated or truncated

    # Compute metrics
    sub_env = env.envs[0].env  # Access the base environment

    # Calculate PSP and LHP components
    lambda_S = (sub_env.mu_S - sub_env.r) / sub_env.sigma_S
    lambda_B = (sub_env.mu_B - sub_env.r) / sub_env.sigma_B

    # Compute PSP components
    pi_PSP_S = (lambda_S / (sub_env.gamma * sub_env.sigma_S * np.sqrt(1 - sub_env.rho_SB**2))) * 10000
    pi_PSP_B = (1 / (sub_env.gamma * sub_env.sigma_B)) * (lambda_B - lambda_S * (sub_env.rho_SB / np.sqrt(1 - sub_env.rho_SB**2))) * 10000

    # Compute LHP components
    pi_LHP_S = ((1 - 1 / sub_env.gamma) * (sub_env.sigma_L * (sub_env.rho_LS - sub_env.rho_LB * sub_env.rho_SB)) / (sub_env.sigma_S * (1 - sub_env.rho_SB**2))) * 10000
    pi_LHP_B = ((1 - 1 / sub_env.gamma) * (sub_env.sigma_L * (sub_env.rho_LB - sub_env.rho_LS * sub_env.rho_SB)) / (sub_env.sigma_B * (1 - sub_env.rho_SB**2))) * 10000


    pi_LHP_S_star = ((sub_env.sigma_L * (sub_env.rho_LS - sub_env.rho_LB * sub_env.rho_SB)) / (sub_env.sigma_S * (1 - sub_env.rho_SB**2)) * 10000)
    pi_LHP_B_star = ((sub_env.sigma_L * (sub_env.rho_LB - sub_env.rho_LS * sub_env.rho_SB)) / (sub_env.sigma_B * (1 - sub_env.rho_SB**2)) * 10000)   

    pi_LHP_S_star_gamma = pi_LHP_S - pi_LHP_S_star
    pi_LHP_B_star_gamma = pi_LHP_B - pi_LHP_B_star 
    
    # Compute πS and πB from PSP + LHP
    pi_S = pi_PSP_S + pi_LHP_S
    pi_B = pi_PSP_B + pi_LHP_B
    pi_C = 100 - (pi_S/100 + pi_B/100)  # Ensure total is 100%

    
    # Calculate ξ (funding-ratio equivalent utility loss)
    x_CwL = (lambda_B**2 + lambda_S**2) / (2 * sub_env.gamma) \
         - ((1 - sub_env.gamma) / sub_env.gamma) * sub_env.sigma_L * \
           (lambda_B * sub_env.rho_LB + lambda_S * (sub_env.rho_LS - sub_env.rho_LB * sub_env.rho_SB) /
            np.sqrt(1 - sub_env.rho_SB**2)) \
         - (sub_env.mu_L - sub_env.r) / sub_env.sigma_L \
         + sub_env.sigma_L**2 * (1 - sub_env.gamma / 2)

    x_CwoL = (lambda_B**2 + lambda_S**2) / (2 * sub_env.gamma) \
         - (sub_env.mu_L - sub_env.r) / sub_env.sigma_L \
         + sub_env.sigma_L**2 * (1 - sub_env.gamma / 2)

    # Debugging: Print intermediate values
    #print(f"x_CwL: {x_CwL}, x_CwoL: {x_CwoL}, Difference: {x_CwL - x_CwoL}")

    #Compute empirical scaling factor to match expected ξ range
    #print(f"x_CwL: {x_CwL}, x_CwoL: {x_CwoL}, Difference: {x_CwL - x_CwoL}")

    difference = abs((x_CwoL - x_CwL) * sub_env.time_horizon)

    xi = max(np.exp(difference) - 1, 0)  # No forced scaling

    # Cap ξ to avoid unrealistic values
    xi = min(xi, 110)


    #print(f"ρLB: {rho_LB}, x_CwL: {x_CwL}, x_CwoL: {x_CwoL}, Difference: {difference}")
    #print(f"Scaling Factor: {scaling_factor}, Raw exp(difference): {np.exp(difference)}, Corrected ξ: {xi}")

    # Append results
    results.append([
        rho_LB, round(pi_S/ 100, 2), round(pi_PSP_S/100, 2), round(pi_LHP_S/ 100, 2),
        round(pi_LHP_S_star / 100, 2), round(pi_LHP_S_star_gamma / 100, 2),
        round(pi_B / 100, 2), round(pi_PSP_B/100, 2), round(pi_LHP_B / 100, 2),
        round(pi_LHP_B_star / 100, 2), round(pi_LHP_B_star_gamma / 100, 2),
        round(pi_C, 2), round(xi, 6),
    ])

# Save results
columns = ["ρLB", "πS", "πPSP_S", "πLHP_S", "πS^LHP*", "πS^LHP*(γ)", 
           "πB", "πPSP_B", "πLHP_B", "πB^LHP*", "πB^LHP*(γ)", 
           "πC", "ξ"]

df_results = pd.DataFrame(results, columns=columns)
df_results.to_csv("RL_ALM_Results_Volatility.csv", index=False)
print(df_results)